In [34]:
import os
import pickle
import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score

In [35]:
# Paths
BASE_DIR = ""  
ARTIFACT_DIR = os.path.join(BASE_DIR, "artifacts/")
RESULTS_DIR = os.path.join(BASE_DIR, "results/")

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Artifact dir:", ARTIFACT_DIR)
print("Results dir:", RESULTS_DIR)

Artifact dir: artifacts/
Results dir: results/


In [ ]:
import joblib

def load_artifact(path):
    return joblib.load(path)

train_df = load_artifact(os.path.join(ARTIFACT_DIR, "train_df.pkl"))
val_df   = load_artifact(os.path.join(ARTIFACT_DIR, "val_df.pkl"))
test_df  = load_artifact(os.path.join(ARTIFACT_DIR, "test_df.pkl"))

Y_train = load_artifact(os.path.join(ARTIFACT_DIR, "Y_train.pkl"))
Y_val   = load_artifact(os.path.join(ARTIFACT_DIR, "Y_val.pkl"))
Y_test  = load_artifact(os.path.join(ARTIFACT_DIR, "Y_test.pkl"))

mlb = joblib.load("label_binarizer.pkl")
label_list = list(mlb.classes_)
label_set = set(label_list)


In [37]:
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (88500, 4)
Val shape: (11063, 4)
Test shape: (11063, 4)


In [38]:
import numpy as np
from sklearn.metrics import f1_score

def evaluate_predictions(y_true, y_pred, threshold=0.5):
    """
    y_true: ground truth (multi-hot)
    y_pred: probabilities OR binary predictions
    """

    # If probabilities → convert to binary
    if y_pred.dtype != np.int32 and y_pred.dtype != np.int64:
        y_pred = (y_pred >= threshold).astype(int)

    micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

    return {
        "micro_f1": micro,
        "macro_f1": macro
    }

In [39]:
def precision_at_k(y_true, y_scores, k=5):
    top_k = np.argsort(-y_scores, axis=1)[:, :k]
    precisions = []

    for i in range(y_true.shape[0]):
        true_set = set(np.where(y_true[i] == 1)[0])
        pred_set = set(top_k[i])
        precisions.append(len(true_set & pred_set) / k)

    return np.mean(precisions)


def recall_at_k(y_true, y_scores, k=5):
    top_k = np.argsort(-y_scores, axis=1)[:, :k]
    recalls = []

    for i in range(y_true.shape[0]):
        true_set = set(np.where(y_true[i] == 1)[0])
        pred_set = set(top_k[i])
        if len(true_set) == 0:
            continue
        recalls.append(len(true_set & pred_set) / len(true_set))

    return np.mean(recalls)

## TF-IDF+Logistic Regression Baseline

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9
)

X_train = tfidf.fit_transform(train_df["text"])
X_val = tfidf.transform(val_df["text"])
X_test = tfidf.transform(test_df["text"])

clf = OneVsRestClassifier(
    LogisticRegression(max_iter=300, class_weight="balanced")
)

clf.fit(X_train, Y_train)

val_probs = clf.predict_proba(X_val)
val_pred = (val_probs >= 0.5).astype(int)

print("Val Macro-F1:", f1_score(Y_val, val_pred, average="macro"))
print("Val Micro-F1:", f1_score(Y_val, val_pred, average="micro"))

In [ ]:
val_probs = clf.predict_proba(X_val)
test_probs = clf.predict_proba(X_test)

In [ ]:
val_metrics = evaluate_predictions(Y_val, val_probs)
test_metrics = evaluate_predictions(Y_test, test_probs)

print("Validation:", val_metrics)
print("Test:", test_metrics)
print("P@5:", precision_at_k(Y_test, test_probs, k=5))
print("R@5:", recall_at_k(Y_test, test_probs, k=5))

In [ ]:
results = []

def log_result(model_name, mode, metrics, p5=None, r5=None):
    results.append({
        "model": model_name,
        "mode": mode,
        "micro_f1": metrics["micro_f1"],
        "macro_f1": metrics["macro_f1"],
        "precision@5": p5,
        "recall@5": r5
    })

log_result(
    "TF-IDF",
    "direct",
    test_metrics,
    precision_at_k(Y_test, test_probs, 5),
    recall_at_k(Y_test, test_probs, 5)
)

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df

## BioBERT

In [ ]:
import numpy as np
import pandas as pd
import random
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

In [ ]:
print(train_df.shape, val_df.shape, test_df.shape)
print(Y_train.shape, Y_val.shape, Y_test.shape)
print(train_df.columns)

In [ ]:
MAX_SEQS = 4
WORDS_PER_SEQ = 32

def make_chunks_from_text(text, max_seqs=4, words_per_seq=32):
    if not isinstance(text, str):
        text = "" if text is None else str(text)

    words = text.split()
    words = words[:max_seqs * words_per_seq]

    chunks = []
    for i in range(0, len(words), words_per_seq):
        chunk = words[i:i + words_per_seq]
        if chunk:
            chunks.append(" ".join(chunk))

    while len(chunks) < max_seqs:
        chunks.append("")

    return chunks[:max_seqs]

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["chunks"] = train_df["text"].apply(lambda x: make_chunks_from_text(x, MAX_SEQS, WORDS_PER_SEQ))
val_df["chunks"] = val_df["text"].apply(lambda x: make_chunks_from_text(x, MAX_SEQS, WORDS_PER_SEQ))
test_df["chunks"] = test_df["text"].apply(lambda x: make_chunks_from_text(x, MAX_SEQS, WORDS_PER_SEQ))

print(train_df["chunks"].iloc[0][:2])
print("Num chunks:", len(train_df["chunks"].iloc[0]))

In [ ]:
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class ICDDataset(Dataset):
    def __init__(self, df, y):
        self.chunks = df["chunks"].tolist()
        self.labels = np.asarray(y, dtype=np.float32)

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        return self.chunks[idx], self.labels[idx]


def collate_fn(batch):
    batch_chunks, batch_labels = zip(*batch)

    B = len(batch_chunks)
    S = len(batch_chunks[0])

    flat_chunks = [chunk for doc in batch_chunks for chunk in doc]

    enc = tokenizer(
        flat_chunks,
        padding=True,
        truncation=True,
        max_length=96,
        return_tensors="pt"
    )

    labels = torch.tensor(np.array(batch_labels), dtype=torch.float32)
    return enc, labels, B, S

In [ ]:
class D2SBERTSequenceAttention(nn.Module):
    def __init__(self, model_name, num_labels, freeze_encoder=False):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size

        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False

        self.seq_attention = nn.Parameter(torch.randn(hidden, num_labels) * 0.02)
        self.classifier_weight = nn.Parameter(torch.randn(num_labels, hidden) * 0.02)
        self.classifier_bias = nn.Parameter(torch.zeros(num_labels))

    def forward(self, encodings, batch_size, num_seqs):
        outputs = self.encoder(**encodings)
        cls = outputs.last_hidden_state[:, 0, :]   # [B*S, H]

        H = cls.size(-1)
        D = cls.view(batch_size, num_seqs, H)      # [B, S, H]

        scores = torch.tanh(torch.einsum("bsh,hc->bsc", D, self.seq_attention))
        alpha = torch.softmax(scores, dim=1)       # attention over sequences

        label_repr = torch.einsum("bsc,bsh->bch", alpha, D)
        logits = (label_repr * self.classifier_weight.unsqueeze(0)).sum(dim=-1) + self.classifier_bias

        return logits

In [ ]:
BATCH_SIZE = 1

train_ds = ICDDataset(train_df, Y_train)
val_ds = ICDDataset(val_df, Y_val)
test_ds = ICDDataset(test_df, Y_test)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

In [ ]:
model = D2SBERTSequenceAttention(
    MODEL_NAME,
    num_labels=Y_train.shape[1],
    freeze_encoder=False
).to(device)

optimizer = AdamW(model.parameters(), lr=1e-5)
criterion = nn.BCEWithLogitsLoss()

scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

In [ ]:
def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for enc, labels, B, S in tqdm(loader, desc="Evaluating", leave=False):
            enc = {k: v.to(device, non_blocking=True) for k, v in enc.items()}
            labels = labels.to(device, non_blocking=True)

            if torch.cuda.is_available():
                with torch.cuda.amp.autocast():
                    logits = model(enc, B, S)
            else:
                logits = model(enc, B, S)

            all_logits.append(logits.detach().cpu())
            all_labels.append(labels.detach().cpu())

    if len(all_logits) == 0:
        return {"micro_f1": 0.0, "macro_f1": 0.0}, None, None

    logits = torch.cat(all_logits, dim=0).numpy()
    labels = torch.cat(all_labels, dim=0).numpy()

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= threshold).astype(np.int32)

    micro = f1_score(labels, preds, average="micro", zero_division=0)
    macro = f1_score(labels, preds, average="macro", zero_division=0)

    return {"micro_f1": micro, "macro_f1": macro}, probs, preds

In [ ]:
EPOCHS = 1
best_val_micro = -1
best_state = None

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for enc, labels, B, S in progress_bar:
        enc = {k: v.to(device, non_blocking=True) for k, v in enc.items()}
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if torch.cuda.is_available():
            with torch.cuda.amp.autocast():
                logits = model(enc, B, S)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(enc, B, S)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = total_loss / max(len(train_loader), 1)
    val_metrics, _, _ = evaluate(model, val_loader)

    print(
        f"Epoch {epoch+1} | "
        f"train_loss={avg_train_loss:.4f} | "
        f"val_micro={val_metrics['micro_f1']:.4f} | "
        f"val_macro={val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["micro_f1"] > best_val_micro:
        best_val_micro = val_metrics["micro_f1"]
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)

print(f"Best val micro-F1: {best_val_micro:.4f}")

In [ ]:
test_metrics, test_probs, test_preds = evaluate(model, test_loader)

print("Test Micro-F1:", round(test_metrics["micro_f1"], 4))
print("Test Macro-F1:", round(test_metrics["macro_f1"], 4))

In [ ]:
torch.save(model.state_dict(), "biobert_icd_top50.pt")
print("Model saved.")